# mask-to-mri — CPU Training Pipeline

Optimized for local CPU training. Runs end-to-end.

**Note:** Set `subset_size` in Cell 2 to control training time (default: 200 slices).

## 1 — Setup & Config

In [1]:
import os, sys
sys.path.insert(0, '.')

import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

# CPU optimizations
torch.set_num_threads(8)
torch.set_num_interop_threads(4)

with open('config.yaml') as f:
    config = yaml.safe_load(f)

from src.utils import fix_seed
fix_seed(config['data']['seed'])
device = torch.device('cpu')
print(f"Device: {device}")
print(f"Threads: {torch.get_num_threads()}")

  → Random seed fixed: 42
Device: cpu
Threads: 8


## 2 — DataLoaders (Auto-generates splits if needed)

In [2]:
import json, os
from pathlib import Path
import tifffile
from src.dataset import LGGDataset, get_patient_file_list, patient_level_split

splits_path = 'data/splits'
dataset_root = config['data']['raw_dir']

# Auto-generate splits if missing
if not all(os.path.exists(f'{splits_path}/{s}_split.json') for s in ['train', 'val', 'test']):
    print("Preparing dataset splits (one-time)...")
    os.makedirs(splits_path, exist_ok=True)
    
    pd_data = get_patient_file_list(dataset_root)
    splits = patient_level_split(pd_data, seed=42)
    
    all_pairs = []
    raw_path = Path(dataset_root)
    for p in sorted(raw_path.iterdir()):
        if p.is_dir():
            for m in sorted(p.glob("*_mask.tif")):
                if tifffile.imread(str(m)).max() > 0:
                    rel_m = str(m.relative_to(raw_path))
                    rel_i = rel_m.replace("_mask.tif", ".tif")
                    all_pairs.append((rel_i, rel_m))
    
    # Assign to splits
    train_pairs, val_pairs, test_pairs = [], [], []
    train_pats = set(list(splits.keys())[0] if False else []) # Not needed, we map by patient
    # Better: Re-use patient data to map paths
    # We will just run the simple scan logic from prepare_splits directly to be safe
    # Actually, simpler: just use the split logic from prepare_splits.py
    !python prepare_splits.py
    print("Splits ready.")

# Load splits
def load_split(name):
    pairs = json.load(open(f'{splits_path}/{name}_split.json'))
    return [(f"{dataset_root}/{p[0]}", f"{dataset_root}/{p[1]}") for p in pairs]

train_pairs = load_split('train')
val_pairs = load_split('val')
test_pairs = load_split('test')

# Subset for CPU
subset_size = 200
if len(train_pairs) > subset_size:
    train_pairs = train_pairs[:subset_size]
    print(f"Training on subset of {len(train_pairs)} pairs.")

print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}, Test: {len(test_pairs)}")

def make_loader(pairs, shuffle, aug):
    ds = LGGDataset(pairs, image_size=256, augment=aug, seed=42, filter_empty_masks=False)
    return torch.utils.data.DataLoader(ds, batch_size=1, shuffle=shuffle, num_workers=0)

loaders = {
    'train': make_loader(train_pairs, shuffle=True, aug=True),
    'val': make_loader(val_pairs, shuffle=False, aug=False),
    'test': make_loader(test_pairs, shuffle=False, aug=False),
}
print("DataLoaders ready.")

Training on subset of 200 pairs.
Train: 200, Val: 151, Test: 157
DataLoaders ready.


c:\Users\pc\Desktop\cours\S6\Vision par Ordinateur\Mask-to-MRI\.venv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


## 3 — Models

In [3]:
from src.generator import create_generator
from src.discriminator import create_discriminator
from src.utils import print_model_summary

m = config['model']
# Uncomment to halve model size for faster CPU training
# m['num_filters'] = 32

G = create_generator(
    in_channels=m['input_channels'],
    out_channels=m['output_channels'],
    num_filters=m['num_filters'],
    norm=m['norm'],
)

D = create_discriminator(
    in_channels=m['input_channels'] + m['output_channels'],
    num_filters=m['num_filters'],
)

print_model_summary('Generator (U-Net)', G)
print_model_summary('Discriminator (PatchGAN)', D)

  Generator (U-Net): 10,750,659 parameters (10.75M)
  Discriminator (PatchGAN): 2,764,865 parameters (2.76M)


## 4 — Training

In [4]:
import time, glob, os
from src.train import train

def find_latest_checkpoint(checkpoint_dir):
    if not os.path.exists(checkpoint_dir): return None
    ckpts = glob.glob(os.path.join(checkpoint_dir, "checkpoint_epoch_*.pt"))
    if not ckpts: return None
    return max(ckpts, key=lambda p: int(p.rsplit("_", 1)[1].split(".")[0]))

ckpt = find_latest_checkpoint(config['paths']['checkpoints'])
if ckpt:
    print(f"⚡ Found checkpoint: {ckpt}")
    print("   → Training will auto-resume from that epoch\n")
else:
    print("No checkpoint found — starting from epoch 1\n")

start = time.time()
try:
    history = train(
        train_loader=loaders['train'],
        val_loader=loaders['val'],
        generator=G,
        discriminator=D,
        config=config,
        device=device,
        checkpoint_dir=config['paths']['checkpoints'],
        samples_dir=config['paths']['samples'],
    )
    print(f"\n✓ Training complete in {time.time()-start:.0f}s")
except KeyboardInterrupt:
    elapsed = time.time() - start
    print(f"\n⏹ Training interrupted after {elapsed:.0f}s")
    print("Checkpoint already saved — resume by re-running this cell")

No checkpoint found — starting from epoch 1


Training pix2pix: epoch 1–200 of 200
  LR=0.0002, beta1=0.5, lambda_L1=100
  Schedule: 100 epochs constant + 100 epochs linear decay
  Checkpoint every 5 epochs



Epoch 1/200 | D: 0.4884 | G: 23.9012 | G_adv: 0.9274 | G_L1: 0.2297 | LR: 0.000200


Epoch 2/200 | D: 0.1066 | G: 17.9454 | G_adv: 0.8719 | G_L1: 0.1707 | LR: 0.000200


Epoch 3/200 | D: 0.0931 | G: 16.9894 | G_adv: 0.9223 | G_L1: 0.1607 | LR: 0.000200


Epoch 4/200 | D: 0.0466 | G: 17.0311 | G_adv: 0.9500 | G_L1: 0.1608 | LR: 0.000200


...

Epoch 10/200 | D: 0.0665 | G: 16.9913 | G_adv: 0.9072 | G_L1: 0.1608 | LR: 0.000200
  → Saved checkpoint: outputs/checkpoints\checkpoint_epoch_10.pt
  → Saved sample grid: outputs/samples\samples_epoch_10.png


Epoch 11/200 | D: 0.0862 | G: 17.0867 | G_adv: 0.9476 | G_L1: 0.1614 | LR: 0.000200


Epoch 12/200 | D: 0.0561 | G: 16.7642 | G_adv: 0.9000 | G_L1: 0.1586 | LR: 0.000200


Epoch 13/200 | D: 0.0674 | G: 17.1415 | G_adv: 0.8922 | G_L1: 0.1625 | LR: 0.000200



⏹ Training interrupted after 1887s
Checkpoint already saved — resume by re-running this cell


## 5 — Results

In [5]:
from src.utils import plot_loss_curves
from src.evaluate import compute_ssim_batch, compute_psnr_batch, save_eval_results
from PIL import Image
from src.train import load_checkpoint

# Load latest checkpoint
ckpts = sorted(glob.glob('outputs/checkpoints/checkpoint_epoch_*.pt'))
if ckpts:
    load_checkpoint(ckpts[-1], G, D)

G.eval()
syn_dir = config['data']['synthetic_dir']
os.makedirs(syn_dir, exist_ok=True)

all_ssim, all_psnr = [], []
count = 0
with torch.no_grad():
    for mask, real in loaders['test']:
        fake = G(mask)
        all_ssim.append(compute_ssim_batch(fake, real))
        all_psnr.append(compute_psnr_batch(fake, real))
        
        fake_np = ((fake[0].permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
        Image.fromarray(fake_np).save(os.path.join(syn_dir, f'fake_{count:04d}.png'))
        count += 1

mean_ssim = np.mean(all_ssim)
mean_psnr = np.mean(all_psnr)
print(f"Generated {count} synthetic slices")
print(f"SSIM: {mean_ssim:.4f} | PSNR: {mean_psnr:.2f} dB")

save_eval_results({'ssim': round(mean_ssim, 4), 'psnr': round(mean_psnr, 2)}, 
                  metrics_dir=config['paths']['metrics'], prefix='eval_exp_a')

plot_loss_curves(history)

Generated 157 synthetic slices
SSIM: 0.5315 | PSNR: 19.20 dB
  → Saved evaluation results: outputs/metrics\eval_exp_a_20260404_230840.json


NameError: name 'history' is not defined